# NB3 - Energy injection and the thermal history
Tonale school, *Multimessenger cosmology*: Lecture 4 of the working-group material.

1. The standard thermal history: free electrons and the gas temperature.
2. A decaying relic: the lifetime selects the redshift, hence the messenger.
3. Annihilation: another history, same calorimeter.
4. The cold gas and the 21 cm line.
5. Before recombination: spectral distortions.
6. **Open question: two CMB data sets, an annihilation and a short-lived decay. Which is which?**

Without `classy` the notebook reads precomputed arrays in `data/nb3.npz`: the plots are the same, but you cannot
change the parameters. All runs assume that the injected energy is deposited on the spot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    D = dict(np.load("data/nb3.npz"))
except FileNotFoundError:
    D = {}
try:                                  # CLASS is used only if it is installed and passes a quick test
    from classy import Class
    _c = Class(); _c.set({"h": 0.6736, "omega_b": 0.02237, "omega_cdm": 0.12, "YHe": 0.2454}); _c.compute()
    HAVE_CLASS = abs(_c.Neff() - 3.044) < 0.05          # some installations of classy return N_eff = 0
except Exception:
    HAVE_CLASS = False
if not HAVE_CLASS:
    print("classy is missing or fails a quick test (N_eff should be 3.044): "
          "using the precomputed arrays of data/nb3.npz")

def get(key, compute):
    "Run CLASS, or read the precomputed array with the same key."
    return np.array(compute()) if HAVE_CLASS else D[key]

plt.rcParams.update({"figure.figsize": (7, 4.5), "font.size": 13})
Z = np.logspace(0, np.log10(3001.0), 300) - 1          # redshifts of the thermal history
ELL = np.arange(2, 2501)

## 0. The sources
Each function returns the CLASS parameters of one source. `history` gives the free-electron fraction $x_e(z)$ and the
gas temperature; `spectra` the CMB temperature and polarisation spectra.

| source | function | parameters |
|---|---|---|
| decay | `decay(tau, f)` | lifetime in s, fraction of the dark matter that decays |
| annihilation | `annihilation(p_ann)` | $p_{\rm ann}=f_{\rm eff}\langle\sigma v\rangle/m$ in cm$^3$/s/GeV |


In [ ]:
BASE = {"h": 0.6736, "omega_b": 0.02237, "omega_cdm": 0.1200, "A_s": 2.1e-9, "n_s": 0.9649,
        "z_reio": 7.67, "N_ur": 3.044}

def decay(tau, f):
    return {"DM_decay_fraction": f, "DM_decay_Gamma": 1.0 / tau}

def annihilation(p_ann):
    return {"DM_annihilation_efficiency": p_ann * 6.24e3}          # CLASS wants m^3/s/J

def run(source=None, output=""):
    p = dict(BASE, output=output)
    if "Cl" in output:
        p["l_max_scalars"] = 2500
    p.update(source or {})
    c = Class(); c.set(p); c.compute()
    return c

def history(source=None):
    "x_e and T_gas [K] on the grid Z."
    th = run(source).get_thermodynamics()
    return np.interp(Z, th["z"], th["x_e"]), np.interp(Z, th["z"], th["Tb [K]"])

def spectra(source=None):
    "C_l^TT and C_l^EE on the grid ELL."
    cl = run(source, output="tCl,pCl").raw_cl(2500)
    return cl["tt"][2:], cl["ee"][2:]

## 1. The standard thermal history
**Look at**: when does $x_e$ freeze out, and at which value? Until when does the gas follow the CMB temperature,
and how does it cool afterwards? Why is a Universe with $x_e\sim2\times10^{-4}$ and a gas at a few kelvin such a
sensitive calorimeter?

In [ ]:
xe0, tg0 = get("history_lcdm", history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
ax[0].loglog(1 + Z, xe0); ax[0].set_xlabel("1 + z"); ax[0].set_ylabel(r"$x_e$")
ax[1].loglog(1 + Z, tg0, label="gas"); ax[1].loglog(1 + Z, 2.7255 * (1 + Z), "k--", label="CMB")
ax[1].set_xlabel("1 + z"); ax[1].set_ylabel("T [K]"); ax[1].legend();

## 2. A decaying relic
Three lifetimes, each with the fraction $f$ allowed by Planck (roughly): $10^{13}$ s (just after recombination),
$10^{15}$ s (the dark ages), $10^{26}$ s (longer than the age of the Universe, here with all the dark matter).

**Look at**: at which redshift does each lifetime act? Why is the allowed fraction so tiny for the short
lifetimes? In the spectra, which multipoles respond to an injection around recombination, and which to a late one
(compare with reionisation)?

In [ ]:
cases = [(1e13, 1e-10), (1e15, 3e-11), (1e26, 1.0)]
tt0, ee0 = get("spectra_lcdm", spectra)
fig, ax = plt.subplots(1, 3, figsize=(18, 4.3))
for tau, f in cases:
    lab = r"$\tau=10^{%d}$ s, $f=%g$" % (np.log10(tau), f)
    xe, tg = get("history_decay_%g" % tau, lambda: history(decay(tau, f)))
    tt, ee = get("spectra_decay_%g" % tau, lambda: spectra(decay(tau, f)))
    ax[0].loglog(1 + Z, xe / xe0, label=lab); ax[1].semilogx(ELL, tt / tt0 - 1); ax[2].semilogx(ELL, ee / ee0 - 1)
ax[0].set_xlabel("1 + z"); ax[0].set_ylabel(r"$x_e/x_e^{\Lambda{\rm CDM}}$"); ax[0].legend(fontsize=10)
for a, t in zip(ax[1:], ("TT", "EE")):
    a.axhline(0, color="k", lw=0.8); a.set_xlabel(r"$\ell$"); a.set_ylabel(r"$\Delta C_\ell^{%s}/C_\ell^{%s}$" % (t, t))

## 3. Annihilation
Three values of $p_{\rm ann}=f_{\rm eff}\langle\sigma v\rangle/m$: below, at and above the Planck limit
($3.2\times10^{-28}\,{\rm cm^3\,s^{-1}\,GeV^{-1}}$). Dashed: the decay with $\tau=10^{13}$ s of section 2.

**Look at**: annihilation follows the density squared and a decay the density: how does that show in $x_e(z)$?
How do $x_e$ and the spectra scale with $p_{\rm ann}$? Which multipoles respond in TT and in EE? Compare with the decay:
could the CMB tell the two apart?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.3))
for p_ann in (1e-28, 3.2e-28, 1e-27):
    xe, tg = get("history_ann_%g" % p_ann, lambda: history(annihilation(p_ann)))
    tt, ee = get("spectra_ann_%g" % p_ann, lambda: spectra(annihilation(p_ann)))
    ax[0].loglog(1 + Z, xe / xe0, label=r"$p_{\rm ann}=%g$" % p_ann)
    ax[1].semilogx(ELL, tt / tt0 - 1); ax[2].semilogx(ELL, ee / ee0 - 1)
    print("p_ann = %7.1e : x_e/x_e(LCDM) = %.2f at z = 600, %.2f at z = 100; TT %+.1f%% at l = 2000; EE %+.1f%% at l = 30"
          % (p_ann, np.interp(600, Z, xe / xe0), np.interp(100, Z, xe / xe0), 100 * (tt / tt0 - 1)[1998], 100 * (ee / ee0 - 1)[28]))
xe, tg = get("history_decay_%g" % 1e13, lambda: history(decay(1e13, 1e-10)))
tt, ee = get("spectra_decay_%g" % 1e13, lambda: spectra(decay(1e13, 1e-10)))
ax[0].loglog(1 + Z, xe / xe0, "k--", lw=1, label=r"decay, $\tau=10^{13}$ s, $f=10^{-10}$")
ax[1].semilogx(ELL, tt / tt0 - 1, "k--", lw=1); ax[2].semilogx(ELL, ee / ee0 - 1, "k--", lw=1)
ax[0].set_xlabel("1 + z"); ax[0].set_ylabel(r"$x_e/x_e^{\Lambda{\rm CDM}}$"); ax[0].legend(fontsize=10)
for a, t in zip(ax[1:], ("TT", "EE")):
    a.axhline(0, color="k", lw=0.8); a.set_xlabel(r"$\ell$"); a.set_ylabel(r"$\Delta C_\ell^{%s}/C_\ell^{%s}$" % (t, t))

## 4. The cold gas and the 21 cm line
Before the first stars, the deepest possible 21 cm absorption is set by the gas temperature:
$\delta T_b\simeq27\,{\rm mK}\,(1-x_e)\,(1-T_{\rm CMB}/T_{\rm gas})\sqrt{(1+z)/10}$ (spin temperature equal to the gas temperature).

**Look at**: how deep is the trough at $z=17$ in $\Lambda$CDM? Which lifetime erases half of it? Compare with the
CMB bound $\tau\gtrsim10^{25}$ s: which messenger reaches further?

In [ ]:
def dtb(xe, tg):
    return 27.0 * (1 - xe) * (1 - 2.7255 * (1 + Z) / tg) * np.sqrt((1 + Z) / 10.0)

sel = (Z > 10.5) & (Z < 40)
plt.plot(Z[sel], dtb(xe0, tg0)[sel], "k", label=r"$\Lambda$CDM")
for tau in (1e26, 1e27, 1e28):
    xe, tg = get("history_long_%g" % tau, lambda: history(decay(tau, 1.0)))
    plt.plot(Z[sel], dtb(xe, tg)[sel], label=r"decay, $\tau=10^{%d}$ s" % np.log10(tau))
    print("tau = %.0e s: T_gas(z=17) = %5.1f K, deepest absorption %5.0f mK" % (tau, np.interp(17, Z, tg), np.interp(17, Z, dtb(xe, tg))))
print("LCDM          : T_gas(z=17) = %5.1f K, deepest absorption %5.0f mK" % (np.interp(17, Z, tg0), np.interp(17, Z, dtb(xe0, tg0))))
plt.xlabel("z"); plt.ylabel(r"deepest absorption $\delta T_b$ [mK]"); plt.legend();

## 5. Before recombination: spectral distortions
Energy released into the plasma before recombination cannot change $x_e$, but it distorts the CMB spectrum: a
chemical potential $\mu$ if it is released at $5\times10^4\lesssim z\lesssim2\times10^6$, a Compton $y$ later, nothing
earlier (full thermalisation). No CLASS here: the estimate of the lecture,
$\mu\simeq1.4\int\mathcal{J}_\mu\,d(\Delta\rho_\gamma/\rho_\gamma)$ and $y\simeq\frac14\int\mathcal{J}_y\,d(\Delta\rho_\gamma/\rho_\gamma)$, with visibility functions.

**Look at**: check your pen-and-paper estimate for $\tau=10^8$ s ($\mu\simeq0.014f$). For which lifetimes is the bound
strongest? Why does it disappear below $10^6$ s? How does FIRAS ($|\mu|<9\times10^{-5}$, $|y|<1.5\times10^{-5}$) compare
with the CMB anisotropy bounds of section 2?

In [ ]:
def distortion(tau, f=1.0):
    "mu and y from a decay into photons (radiation era, t = 2.4e19 s / (1+z)^2)."
    z = np.logspace(2.5, 7.5, 3000); t = 2.4e19 / (1 + z)**2
    dq = f * (0.12 / 2.47e-5) / (1 + z) * 2 * (t / tau) * np.exp(-t / tau)      # d(Delta rho / rho) / d ln z
    j_bb = np.exp(-(z / 1.98e6)**2.5)                                            # survives thermalisation
    j_y = 1.0 / (1 + ((1 + z) / 6.0e4)**2.58)
    integ = lambda g: np.sum(0.5 * (g[1:] + g[:-1]) * np.diff(np.log(z)))
    return 1.401 * integ(j_bb * (1 - j_y) * dq), 0.25 * integ(j_y * dq)

print("tau = 1e8 s: mu = %.4f f, y = %.5f f" % distortion(1e8))
taus = np.logspace(5, 12.5, 60)
mu, y = np.array([distortion(t) for t in taus]).T
plt.loglog(taus, 9e-5 / mu, label=r"FIRAS, $\mu$"); plt.loglog(taus, 1.5e-5 / y, label=r"FIRAS, $y$")
plt.ylim(1e-4, 10); plt.xlabel(r"lifetime $\tau$ [s]"); plt.ylabel("largest decaying fraction f"); plt.legend();

## 6. Open question: an annihilation and a short-lived decay. Which is which?
Two universes, **X** and **Y**. In one the dark matter annihilates; in the other a small fraction of it decays with a
short lifetime. You are given **measurements** of the temperature and polarisation spectra, $C_\ell^{TT}$ and
$C_\ell^{EE}$, relative to $\Lambda$CDM: an ideal experiment, limited by cosmic variance on 60% of the sky, in 34 bins
of multipoles. All the other cosmological parameters are those of the reference model.

1. Which is which? Look at TT and EE together before computing anything (sections 2 and 3).
2. Fit the parameters: $p_{\rm ann}$ for one, the lifetime and the fraction for the other. `chi2(u, source)` compares a
   source with the data of universe `u` (68 points): scan it, as in the example below.
3. Can the *other* kind of source also fit each universe? What does that say about what the CMB measures?
4. Which other messenger would settle it?

In [ ]:
EDGES = D["myst_edges"]                                # the multipole bins: [EDGES[i], EDGES[i+1])
LC = np.sqrt(EDGES[:-1] * (EDGES[1:] - 1.0))           # their centres

def binned(ratio):
    "A spectrum ratio given on ELL, averaged in the bins of the data."
    return np.array([ratio[(ELL >= lo) & (ELL < hi)].mean() for lo, hi in zip(EDGES[:-1], EDGES[1:])])

def chi2(u, source):
    "chi2 of a source against the TT and EE data of universe u (needs classy)."
    tt, ee = spectra(source)
    return (np.sum(((binned(tt / tt0) - D["myst_%s_tt" % u]) / D["myst_err"])**2)
            + np.sum(((binned(ee / ee0) - D["myst_%s_ee" % u]) / D["myst_err"])**2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for u, c in zip("XY", ("C0", "C3")):
    for a, t in zip(ax, ("tt", "ee")):
        a.errorbar(LC, D["myst_%s_%s" % (u, t)] - 1, D["myst_err"], fmt="o", ms=3, color=c, label="universe " + u)
# overlay and test your sources here, for example (it is not the answer):
# tt, ee = spectra(annihilation(1e-28)); ax[0].plot(ELL, tt / tt0 - 1); ax[1].plot(ELL, ee / ee0 - 1)
# for p_ann in (1e-28, 2e-28): print(p_ann, chi2("X", annihilation(p_ann)))
for a, t, lim in zip(ax, ("TT", "EE"), ((-0.04, 0.02), (-0.06, 0.22))):
    a.set_xscale("log"); a.set_xlim(20, 2600); a.set_ylim(*lim); a.axhline(0, color="k", lw=0.8)
    a.set_xlabel(r"$\ell$"); a.set_ylabel(r"$\Delta C_\ell^{%s}/C_\ell^{%s}$" % (t, t))
ax[0].legend()
print("chi2 of LCDM (no injection): X %.0f, Y %.0f, for %d points" % tuple(
    [sum(np.sum(((D["myst_%s_%s" % (u, t)] - 1) / D["myst_err"])**2) for t in ("tt", "ee")) for u in "XY"] + [2 * len(LC)]))